# implementing LangChain Core

In [1]:
# Implementing an LLMs
import random
class FakeLLM():
    def __init__(self):
        print("LLM Initialized")

    def predict(self, prompt):
        response_list = [
            "Delhi is capital of India",
            "Paris is capital of France",
            "Tokyo is capital of Japan",
            "Abra ka Dabra Gilli- Gilli Chupa Chups"
        ]
        return ({"response": random.choice(response_list), "prompt": prompt, "source": "Fake LLM"})


In [2]:
glue_llm = FakeLLM()

LLM Initialized


In [3]:
glue_llm.predict("What is the capital of France?")

{'response': 'Paris is capital of France',
 'prompt': 'What is the capital of France?',
 'source': 'Fake LLM'}

In [4]:
glue_llm.predict("What is the capital of France?")["response"]

'Abra ka Dabra Gilli- Gilli Chupa Chups'

In [5]:
# Template class
class FakePromptTemplate():
    def __init__(self, template, input_variables ):
        self.template = template
        self.input_variables = input_variables

    def format(self,input_dict):
        return self.template.format(**input_dict)
    def __str__(self):
        return {"prompt is": self.template}


In [6]:
# lets build a template by using FakeTemplateClass
glue_prompt = FakePromptTemplate(
    template = "What is the capital of {country}?",
    input_variables = ["country"]
)

In [7]:
print(glue_prompt.format({"country": "India"}))

What is the capital of India?


In [8]:
# Now lets try to memic the Actual langchain as we have llm and template lets try to generate the response from LLm with prompt
glue_llm= FakeLLM()
glue_prompt= FakePromptTemplate(
    template = " What is a {Topic}",
    input_variables = ["Topic"]
)
# lets pass the prompt to llm
glue_llm.predict(glue_prompt.format({"Topic": "Country"}))["response"]

LLM Initialized


'Paris is capital of France'

In [9]:
class FakeLLmChain:
    def __init__(self, llm, prompt):
        self.llm = llm
        self.prompt = prompt

    def run(self,input_dict):
        final_prompt = self.prompt.format(input_dict)
        result = self.llm.predict(final_prompt)
        return result["response"]

In [10]:
chain = FakeLLmChain(glue_llm, glue_prompt)
chain.run({"Topic": "India"})

'Delhi is capital of India'

# Runnables


In [11]:
from abc import ABC, abstractmethod
class Runnable(ABC):
    @abstractmethod
    def invoke(self, input_data):
        pass

# FakeLLM
class FakeLLM(Runnable):

    def __init__(self):
        print("LLM Initialized")

    def predict(self, prompt):
        response_list = [
            "Delhi is capital of India",
            "Paris is capital of France",
            "Tokyo is capital of Japan",
            "Abra ka Dabra Gilli- Gilli Chupa Chups"
        ]
        return ({"response": random.choice(response_list), "prompt": prompt, "source": "Fake LLM"})
    def invoke(self, prompt):
        response_list = [
            "Delhi is capital of India",
            "Paris is capital of France",
            "Tokyo is capital of Japan",
            "Abra ka Dabra Gilli- Gilli Chupa Chups"
        ]
        return ({"response": random.choice(response_list), "prompt": prompt, "source": "Fake LLM"})

# FakePromptTemplate
class FakePromptTemplate(Runnable):
    def __init__(self, template, input_variables ):
        self.template = template
        self.input_variables = input_variables
    def format(self,input_dict):
        return self.template.format(**input_dict)
    def invoke(self,input_dict):
        return self.format(input_dict)


In [12]:
# Tesing
glen_llm = FakeLLM()
glen_prompt = FakePromptTemplate(
    template = "What is the capital of {country}?",
    input_variables = ["country"]
)
result = glen_llm.invoke(glen_prompt.invoke({"country": "India"})) # here we have to use invoke method twice

LLM Initialized


In [13]:
print(result["response"])

Paris is capital of France


In [14]:
# To avoid the multiple usages of invoke
class RunnableConnector(Runnable):
    def __init__(self,runnable_list):
        self.runnable_list = runnable_list

    def invoke(self, input_data):
        for runable in self.runnable_list:
            input_data = runable.invoke(input_data)
        return input_data



In [15]:
glen_llm = FakeLLM()
glen_prompt = FakePromptTemplate(
    template = "What is the capital of {country}?",
    input_variables = ["country"]
)
chain = RunnableConnector([glen_prompt, glen_llm])
chain.invoke({"country": "India"})

LLM Initialized


{'response': 'Tokyo is capital of Japan',
 'prompt': 'What is the capital of India?',
 'source': 'Fake LLM'}

In [16]:
gloss_template = FakePromptTemplate(
    template = "What is a {Topic}",
    input_variables = ["Topic"]
)
glen_llm = FakeLLM()
chain = RunnableConnector(
    [gloss_template, glen_llm]
)
chain.invoke({"Topic": "Country"})

LLM Initialized


{'response': 'Tokyo is capital of Japan',
 'prompt': 'What is a Country',
 'source': 'Fake LLM'}

In [17]:
# Now lets create a Dummy parser
class FakeParser(Runnable):
    def __init__(self):
        pass

    def invoke(self,input_data):
        return input_data["response"]

In [18]:
# now lets use Fake Template, Fakellm, FakeParser
gloss_template =FakePromptTemplate(
    template = "What is a {Topic}",
    input_variables = ["Topic"]
)
gloss_llm = FakeLLM()
gloss_parser = FakeParser()
chain = RunnableConnector([gloss_template, gloss_llm, gloss_parser])
chain.invoke({"Topic": "Country"})

LLM Initialized


'Abra ka Dabra Gilli- Gilli Chupa Chups'

# Types of Runnables
## Task Specific Runnables - These are core langchain component, that has been converted into Runnables so they can be used in pipelines.(ex - LLM calls, Prompt, Retrival, Parser etc)
## Runnable Primitives  -  These are fundamental building blocks, for structuring execution logic in AI workflows. They orchestrate execution by defining how different Runnable will interact(ex - Paralell, Sequeuntially, Conditionally)


# RunnableSequence (a primitive Runnables)
## a sequence of runnable chain that execute, one task after another
 ### T1 >> T2

In [21]:
from langchain_google_genai import GoogleGenerativeAI
from langchain_core.runnables import RunnableSequence
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
from dotenv import load_dotenv
load_dotenv()
gemini_key = os.getenv("GEMINI_API_KEY")
gemini_powered_llm = GoogleGenerativeAI(
    model= "gemini-2.5-flash-lite",
    temperature=0,
    google_api_key=gemini_key
)
prompt = PromptTemplate(
    template = "What is the capital of {country}?",
    input_variables = ["country"]
)
parser =StrOutputParser()
chain = RunnableSequence(prompt, gemini_powered_llm, parser ) # setting them to execute in sequence
result = chain.invoke({"country": "India"})
print(result)

The capital of India is **New Delhi**.


# Runnable Paralell